## Lab 8: Method Chaining and `Super()`

### Method Chaining in Python

<font size = "3">

In Lecture 17, we learned how to work with pandas DataFrames. One thing we can do with pandas is use multiple methods/attributes together in one line of code. This is called **chaining**.

For example, instead of writing separate lines to filter data, group it, compute an average, sort in descending order and extract the the top 5, we can combine those steps into one line.

This works because each method returns something that the next method can use.

Let's try an example. We'll start by chaining a few pandas methods together, then go through each step separately to see what type of object each method returns.

In [1]:
import pandas as pd

df_results = pd.read_csv("data/results.csv")

top_5_teams = df_results.query("raceId >= 500").groupby("constructorId")["points"].mean().sort_values(ascending=False).iloc[:5]

display(top_5_teams)

constructorId
131    12.921905
9      11.412338
6       4.820082
208     4.584416
211     3.723684
Name: points, dtype: float64

<font size = "3">

Now let's split that up into multiple lines, printing the type after each step:

In [2]:
df_results = pd.read_csv("data/results.csv")

print("Original DataFrame:", type(df_results))

step1 = df_results.query("raceId >= 500")
print("After query:", type(step1))

step2 = step1.groupby("constructorId")["points"]
print("After groupby:", type(step2))

step3 = step2.mean()
print("After mean:", type(step3))

step4 = step3.sort_values(ascending=False)
print("After sort:", type(step4))

step5 = step4.iloc[:5]
print("After iloc:", type(step5))

Original DataFrame: <class 'pandas.DataFrame'>
After query: <class 'pandas.DataFrame'>
After groupby: <class 'pandas.api.typing.SeriesGroupBy'>
After mean: <class 'pandas.Series'>
After sort: <class 'pandas.Series'>
After iloc: <class 'pandas.Series'>


### Recall Lab 7: `OrdinaryLeastSquares` class

<font size = "3">

The original `.fit()` method stores coefficients on the object but returns nothing. Calling `.fit()` followed by a call to `.predict()` on a separate line works.

In [3]:
import numpy as np
import numpy.linalg as la

# Functions for our regression model
f_0 = lambda x: 1.0
f_1 = lambda x: np.cos(x)
f_2 = lambda x: np.sin(x)

# Generate sample data
x_vals = np.linspace(-np.pi, np.pi, 20)
y_vals = -1 + 2 * np.cos(x_vals) + np.sin(x_vals)

In [4]:
class OrdinaryLeastSquares:
    def __init__(self, function_list):
        self.f_list = function_list
        self.coefficients = None
        self.x = None
        self.y = None

    def fit(self, x, y):
        self.x = x
        self.y = y
        m, n = len(x), len(self.f_list)
        A = np.zeros((m, n))
        for k in range(n):
            A[:, k] = self.f_list[k](x)
        normal_matrix = A.T @ A
        rhs = A.T @ y
        self.coefficients = la.solve(normal_matrix, rhs)

    def predict(self, x):
        y_pred = np.zeros(x.shape)
        for k in range(len(self.f_list)):
            c = self.coefficients[k]
            f = self.f_list[k](x)
            y_pred += c*f
        return y_pred

<font size = "3">

The next cell runs with no issues:


In [5]:
model = OrdinaryLeastSquares([f_0, f_1, f_2])   # initialize model
model.fit(x_vals, y_vals)                       # fit the coefficients
predictions = model.predict(x_vals)             # use model to predict at new points

print(predictions)

[-3.         -3.21633395 -3.19249373 -2.93106279 -2.46037124 -1.8314258
 -1.11238248 -0.38116077  0.28300011  0.80812802  1.1373172   1.2348949
  1.09028705  0.71916418  0.16174318 -0.52157071 -1.25672984 -1.96406831
 -2.56693501 -3.        ]


<font size = "3">

But if we run the following cell, attempting to chain the calculations together, it raises an `AttributeError`:

In [6]:
model = OrdinaryLeastSquares([f_0, f_1, f_2])

predictions = model.fit(x_vals, y_vals).predict(x_vals)

AttributeError: 'NoneType' object has no attribute 'predict'

<font size = "3">

If we break it into separate steps, we find the culprit: The output of the `.fit` method is `None`.

In [7]:
# break into steps
model = OrdinaryLeastSquares([f_0, f_1, f_2])
print(type(model))
model_fit = model.fit(x_vals, y_vals)
print(type(model_fit))
predictions = model_fit.predict(x_vals)

<class '__main__.OrdinaryLeastSquares'>
<class 'NoneType'>


AttributeError: 'NoneType' object has no attribute 'predict'

### A chaining-compatible `fit()`

<font size = "3">

Below, we'll redefine the class and add `return self` at the end of the `fit()` method. This allows us to call `.predict()` right after `.fit()` without needing a separate line of code.

In [8]:
class OrdinaryLeastSquares:
    def __init__(self, function_list):
        self.f_list = function_list
        self.coefficients = None
        self.x = None
        self.y = None

    def fit(self, x, y):
        self.x = x
        self.y = y
        m, n = len(x), len(self.f_list)
        A = np.zeros((m, n))
        for k in range(n):
            A[:, k] = self.f_list[k](x)
        normal_matrix = A.T @ A
        rhs = A.T @ y
        self.coefficients = la.solve(normal_matrix, rhs)
        return self      # <--------- just need to add this line

    def predict(self, x):
        y_pred = np.zeros(x.shape)
        for k in range(len(self.f_list)):
            c = self.coefficients[k]
            f = self.f_list[k](x)
            y_pred += c*f
        return y_pred

<font size = "3">

So there's no problems with the next cell:

In [9]:
model = OrdinaryLeastSquares([f_0, f_1, f_2])

predictions = model.fit(x_vals, y_vals).predict(x_vals)

print("Predictions:", predictions)

Predictions: [-3.         -3.21633395 -3.19249373 -2.93106279 -2.46037124 -1.8314258
 -1.11238248 -0.38116077  0.28300011  0.80812802  1.1373172   1.2348949
  1.09028705  0.71916418  0.16174318 -0.52157071 -1.25672984 -1.96406831
 -2.56693501 -3.        ]


### Reusing initialization with `super()`

<font size = "3">

In Lab 7, the `SimpleLinearRegression` subclass initialized several attributes manually. Here is the old initializer, shown for comparison:

In [10]:
class SimpleLinearRegression(OrdinaryLeastSquares):
    def __init__(self):
        self.coefficients = None
        self.intercept = None
        self.slope = None

        f_0 = lambda x: 1.0
        f_1 = lambda x: x

        self.f_list = [f_0, f_1]
        self.r_squared = None

    def compute_r_squared(self):
        if self.r_squared is None:
            x_bar = np.mean(self.x)
            y_bar = np.mean(self.y)
            s_xx = sum((self.x - x_bar)**2)
            s_xy = sum((self.x - x_bar)*(self.y - y_bar))
            s_yy = sum((self.y - y_bar)**2)
            self.r_squared = s_xy**2/(s_xx*s_yy)
        return self.r_squared

    

The `OrdinaryLeastSquares` class has its own `__init__` method that we didn't utilize. We can use `super()` to access the parent class' `__init__` method, without worrying about a name conflict (we are using two different functions with the same name). 

Here's the `__init__` method for the parent class:

```python
class OrdinaryLeastSquares:
    def __init__(self, function_list):
        self.f_list = function_list
        self.coefficients = None
        self.x = None
        self.y = None
```

<font size = "3">

We can call the parent class' initializer by executing `super().__init__` with the appropriate arguments:

In [11]:
class SimpleLinearRegression(OrdinaryLeastSquares):
    def __init__(self):
        f_0 = lambda x: 1.0
        f_1 = lambda x: x

        super().__init__([f_0, f_1]) # This calls OrdinaryLeastSquares.__init__() with function_list=[f_0, f_1]
        # The parent class' .__init__ will then define self.f_list, self.coefficients, self.x, self.y

        # Now, we just define the new attribute
        self.r_squared = None

    def compute_r_squared(self):
        if self.r_squared is None:
            x_bar = np.mean(self.x)
            y_bar = np.mean(self.y)
            s_xx = sum((self.x - x_bar)**2)
            s_xy = sum((self.x - x_bar)*(self.y - y_bar))
            s_yy = sum((self.y - y_bar)**2)
            self.r_squared = s_xy**2/(s_xx*s_yy)
        return self.r_squared

In [12]:
linear_x = np.array([1, 2, 3])
linear_y = np.array([0, 3, 2])

L = SimpleLinearRegression()
L.fit(linear_x, linear_y)
print("Intercept and slope:", L.coefficients)
print("R-squared:", L.compute_r_squared())

Intercept and slope: [-0.33333333  1.        ]
R-squared: 0.4285714285714286


### Putting it all together: a Model Builder

<font size = "3">

To practice chaining and `super()` together, we'll write a small "model builder": an object that collects the choices you make before you train a machine learning model. We'll keep the machine learning ideas very simple. Two words are all you need:

- A **feature** is a piece of information the model is allowed to look at, such as `"hours_studied"` or `"hours_slept"`.
- A **hyperparameter** is a setting that *you* choose, which controls how the model behaves. You store it under a name, like `"min_samples"`, along with its value.

Different kinds of models use features and hyperparameters in different ways. The *setup* step is the same for all of them, though: pick some features and set some hyperparameters. That shared setup goes in a parent class. Each kind of model is then defined as a subclass of the parent.

### The parent class: `ModelBuilder`

<font size = "3">

Read through the class below. Notice that **every method ends with `return self`**, just like the chaining-compatible `.fit()` from earlier. So each method call gives back the same builder object, and you can call the next method on it right away:

```python
builder = ModelBuilder()
builder.add_feature("age").add_feature("income").set_hyperparameter("speed", 0.1).show()
```

Without `return self`, we would be attempting to call `.add_feature("income")` on `None`, and we'd get the same `AttributeError` we saw with the original `fit()`.

In [ ]:
class ModelBuilder:
    def __init__(self):
        self.features = []             # features (start as empty list)
        self.hyperparameters = {}      # hyperparameters (start as empty dictionary)

    def add_feature(self, feature):
        self.features.append(feature)
        return self

    def set_hyperparameter(self, name, value):
        self.hyperparameters[name] = value
        return self

    def show(self):
        print("Features:", self.features)
        print("Hyperparameters:", self.hyperparameters)
        return self

### A subclass: `DecisionTreeBuilder`

<font size = "3">

A **decision tree** makes predictions by asking a series of yes/no questions about the features. For example: "Did this student study more than 3 hours?" Every answer sends you down a branch to either another question or a final answer.

<img src="data/images.png" width="400" height="300">

<font size = "3">

The tree starts at the **root node** (the first question) and ends at **leaf nodes** (the predictions).

Our version is even simpler. It asks **only one question**, which has the form "is `feature > threshold`?" To *train* it, we try every feature and every possible threshold. Then we keep the one rule that gets the most training examples right.

We use one hyperparameter, `min_samples`. It sets the minimum number of training examples that must land on *each side* of the question. If `min_samples` is 3, a question that puts only 2 students on one side is not allowed.

None of these machine learning details are the main point of this lab. The focus is how the class is put together.

### Your task: finish `DecisionTreeBuilder`

<font size = "3">

The code cell below has a partly written `DecisionTreeBuilder` class. It inherits from `ModelBuilder`. **Do not change `ModelBuilder`.**

The `train` method is already written for you. You need to write the other four methods: `__init__`, `add_feature`, `predict`, and `show`. Replace each `# YOUR CODE HERE` and `pass` with your own code.

Two rules apply to all of your methods:
- If a method has the same name as a method in `ModelBuilder`, use `super()` to call the parent's version instead of copying its code.
- Unless the instructions say otherwise, end the method with `return self`, so calls can be chained.

---

**1. `__init__(self)`** should do the following:
- Call the parent class's `__init__` using `super()`. This creates `self.features` and `self.hyperparameters`.
- Set the hyperparameter "min_samples" to 1. This is the default. *Hint:* use the inherited `set_hyperparameter` method.
- Create two new attributes and set both to `None`:
  - `self.rule` will later store the learned rule as a tuple `(feature_index, threshold)`.
  - `self.train_accuracy` will later store the accuracy from training.
- `__init__` should not have a return statement.

**2. `add_feature(self, feature)`** should do the following:
- If `feature` is already in `self.features`, print the string "'{feature}' already added, skipping" (for example, "'hours_studied' already added, skipping") and do **not** add it to the list of features again.
- Otherwise, add the feature by calling the parent's `add_feature` using `super()`.
- Make sure the method returns `self` in **both** cases.

**3. `predict(self, row)`** should do the following:
- The input argument `row` will be a list of feature values, like `[4, 4]`.
- `self.rule` is a tuple `(col, threshold)`. Unpack it into two variables.
- Return 1 if `row[col] > threshold`. Otherwise, return 0.
- This method returns a number, **not** `self`.

**4. `show(self)`** should do the following:
- First, call the parent's `show` using `super()`. This prints the features and hyperparameters.
- Then, if `self.rule` is not `None`, unpack it into `col, threshold` and print these two lines:
  ```text
  Learned rule: predict 1 if {feature name} > {threshold}
  Training accuracy: {accuracy}
  ```
  where "{feature name}" is `self.features[col]` and "{accuracy}" is `self.train_accuracy`.
- If `self.rule` is `None`, print the string "Not trained yet (or no valid split found)" instead.
- Return `self`.

In [ ]:
class DecisionTreeBuilder(ModelBuilder):
    def __init__(self):
        # YOUR CODE HERE
        pass

    def add_feature(self, feature):
        # YOUR CODE HERE
        pass

    # ----- train is already written for you: do not change it -----
    def train(self, rows, labels):
        """Find the best single 'feature > threshold' rule.
        Both sides of the split must contain at least min_samples rows."""
        min_n = self.hyperparameters["min_samples"]
        best_acc = -1
        self.rule = None
        for col in range(len(self.features)):
            for threshold in sorted(set(row[col] for row in rows)):
                n_above = sum(row[col] > threshold for row in rows)
                if min(n_above, len(rows) - n_above) < min_n:
                    continue   # split too lopsided, skip it
                guesses = [1 if row[col] > threshold else 0 for row in rows]
                acc = sum(g == y for g, y in zip(guesses, labels)) / len(labels)
                if acc > best_acc:
                    best_acc, self.rule = acc, (col, threshold)
        self.train_accuracy = best_acc
        return self
    # ---------------------------------------------------------------

    def predict(self, row):
        # YOUR CODE HERE
        pass

    def show(self):
        # YOUR CODE HERE
        pass

### Testing your class

<font size = "3">

Each test below makes a new tree and adds features with a chain. In each chain, `"hours_studied"` is added twice on purpose, so you can check that your `add_feature` skips the duplicate without breaking the chain. Then we train and show the result in a single chained line.

**Test 1:** In this data, the students who studied more passed. If your class is correct, this cell prints:

```text
Adding features:
'hours_studied' already added, skipping

Training the model:
Features: ['hours_studied', 'hours_slept']
Hyperparameters: {'min_samples': 1}
Learned rule: predict 1 if hours_studied > 3
Training accuracy: 1.0

Prediction for 4 hours of study, 4 hours of sleep:
Will pass
```

If you get an `AttributeError: 'NoneType' object has no attribute ...`, one of your methods is missing `return self`.

In [ ]:
tree = DecisionTreeBuilder()
print("Adding features:")
tree.add_feature("hours_studied").add_feature("hours_slept").add_feature("hours_studied")

print()
print("Training the model:")
# Student 0: 1 hour of study, 8 hours of sleep
# Student 1: 2 hours of study, 6 hours of sleep, etc.
rows   = [[1, 8], [2, 6], [3, 7], [5, 5], [6, 6], [8, 7]]

# Students 0, 1, 2 failed
# Students 3, 4, 5 passed
labels = [0, 0, 0, 1, 1, 1]   # passed the exam?

predict = tree.train(rows, labels).show().predict([4, 4])
print()
print("Prediction for 4 hours of study, 4 hours of sleep:")
if predict == 1:
    print("Will pass")
else:
    print("Will fail")

<font size = "3">

**Test 2:** We use the same code with different data. This time, no `hours_studied` cutoff separates passing from failing, but `hours_slept` does. The tree should pick the other feature, and the prediction for 4 hours of study and 4 hours of sleep should change.

In [ ]:
tree = DecisionTreeBuilder()
print("Adding features:")
tree.add_feature("hours_studied").add_feature("hours_slept").add_feature("hours_studied")

print()
print("Training the model:")
# Student 0: 5 hour of study, 4 hours of sleep
# Student 1: 6 hours of study, 5 hours of sleep, etc.
rows   = [[5, 4], [6, 5], [2, 8], [1, 7], [7, 8], [3, 9]]

# Students 0, 1 failed
# Students 2, 3, 4, 5 passed
labels = [0, 0, 1, 1, 1, 1]   # passed the exam?

predict = tree.train(rows, labels).show().predict([4, 4])
print()
print("Prediction for 4 hours of study, 4 hours of sleep:")
if predict == 1:
    print("Will pass")
else:
    print("Will fail")

<font size = "3">

**Test 3:** We use the same data as Test 2, but first we change the hyperparameter to `min_samples = 3`. The perfect rule from Test 2 put only 2 students on one side, so it is no longer allowed. The tree must settle for a different rule with lower training accuracy.

Notice that we changed a setting by calling an inherited method, `set_hyperparameter`, on a `DecisionTreeBuilder` object. Since `set_hyperparameter` returns `self`, we could also have added it to the end of the `add_feature` chain on the line above.

In [ ]:
tree = DecisionTreeBuilder()
print("Adding features:")
tree.add_feature("hours_studied").add_feature("hours_slept").add_feature("hours_studied")
tree.set_hyperparameter("min_samples", 3)

print()
print("Training the model:")
# Student 0: 5 hour of study, 4 hours of sleep
# Student 1: 6 hours of study, 5 hours of sleep, etc.
rows   = [[5, 4], [6, 5], [2, 8], [1, 7], [7, 8], [3, 9]]

# Students 0, 1 failed
# Students 2, 3, 4, 5 passed
labels = [0, 0, 1, 1, 1, 1]   # passed the exam?

predict = tree.train(rows, labels).show().predict([4, 4])
print()
print("Prediction for 4 hours of study, 4 hours of sleep:")
if predict == 1:
    print("Will pass")
else:
    print("Will fail")

### Check your understanding

<font size = "3">

1. Suppose your `add_feature` returned `self` when it skipped a duplicate, but returned nothing when it added a new feature. Which line in the tests would fail, and with what error?
2. Suppose you left out the `super().__init__()` call in `DecisionTreeBuilder.__init__`. What error would you get when you create a `DecisionTreeBuilder()`, and why?
3. Why is it reasonable for `predict` to return a number instead of `self`?